# Notebook 10 — Comparateur de traces

## Métrique : F1-score géographique

Pour deux traces A et B, échantillonnées en points :

- **Précision** = % de points de A qui sont à < seuil m d'un point de B
- **Rappel** = % de points de B qui sont à < seuil m d'un point de A
- **F1** = 2 × (P × R) / (P + R)

Métrique symétrique entre 0 et 100%. Seuil par défaut : 50m.

## Cas d'usage

1. Comparer un itinéraire calculé à une trace club spécifique
2. Comparer un itinéraire calculé à l'ensemble des traces club
3. Comparer 2 traces GPX uploadées par l'utilisateur

## Implémentation efficace

Pour 2 traces de 1000 points chacune, ça fait 1M comparaisons. On utilise :
- KDTree (sklearn) pour les recherches de plus proche voisin → O(N log M)
- Reprojection en Lambert 93 pour distances en mètres directs

Temps typique : 0.1 seconde par comparaison de 2 traces.


## 1. Setup

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import KDTree
from sqlalchemy import create_engine, text
from shapely import wkt
import xml.etree.ElementTree as ET
import warnings
warnings.filterwarnings("ignore")

DB_CONFIG = {"user": "postgres", "password": "4421",
             "host": "localhost", "port": 5432, "database": "velo_club"}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)
print("OK")

OK


## 2. Reprojection lat/lon → mètres

In [4]:
def latlon_to_meters(coords):
    """Reprojection approximative WGS84 → mètres (Lambert 93 simplifié).
    
    Pour des distances < 100km en IdF, c'est précis à 0.1%.
    Plus rapide que d'utiliser pyproj pour des bulk ops.
    """
    if len(coords) == 0:
        return np.zeros((0, 2))
    arr = np.array(coords)  # (N, 2) = (lat, lon)
    # Centre IdF approximatif
    LAT_REF = 48.8
    lat_rad = np.radians(LAT_REF)
    # x = lon * cos(lat) * 111320, y = lat * 111320
    x = arr[:, 1] * 111320 * np.cos(lat_rad)
    y = arr[:, 0] * 111320
    return np.column_stack([x, y])


def f1_geo(coords_a, coords_b, threshold_m=50):
    """Calcule le F1-score géographique entre 2 traces.
    
    Args:
        coords_a, coords_b: listes de (lat, lon)
        threshold_m: distance max pour considérer un point "couvert"
    
    Returns:
        dict avec precision, recall, f1, n_a, n_b
    """
    if not coords_a or not coords_b:
        return {"precision": 0, "recall": 0, "f1": 0, "n_a": len(coords_a), "n_b": len(coords_b)}
    
    pts_a = latlon_to_meters(coords_a)
    pts_b = latlon_to_meters(coords_b)
    
    tree_b = KDTree(pts_b)
    dists_a, _ = tree_b.query(pts_a, k=1)
    precision = float((dists_a.flatten() < threshold_m).mean())
    
    tree_a = KDTree(pts_a)
    dists_b, _ = tree_a.query(pts_b, k=1)
    recall = float((dists_b.flatten() < threshold_m).mean())
    
    f1 = 2 * precision * recall / max(precision + recall, 1e-10)
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_a": len(coords_a),
        "n_b": len(coords_b),
    }


# Test
print("Test F1 géo :")
trace_1 = [(48.85, 2.35), (48.86, 2.36), (48.87, 2.37)]
trace_2 = [(48.85, 2.35), (48.86, 2.36), (48.88, 2.40)]  # divergent à la fin
r = f1_geo(trace_1, trace_2, threshold_m=200)
print(f"  precision={r['precision']:.2f}, recall={r['recall']:.2f}, f1={r['f1']:.2f}")

Test F1 géo :
  precision=0.67, recall=0.67, f1=0.67


## 3. Charger les traces club

In [5]:
def load_club_traces():
    """Charge toutes les traces club depuis la DB sous forme de listes de coords."""
    print("Chargement des traces club...")
    with engine.connect() as conn:
        traces_df = pd.read_sql(text("""
            SELECT trace_id, name, ST_AsText(geom) AS wkt_geom
            FROM traces
            WHERE geom IS NOT NULL;
        """), conn)
    
    club_traces = []
    for _, row in traces_df.iterrows():
        try:
            g = wkt.loads(row["wkt_geom"])
            # Gestion MultiLineString
            if hasattr(g, "geoms"):
                coords = []
                for line in g.geoms:
                    coords.extend([(y, x) for x, y in line.coords])
            else:
                coords = [(y, x) for x, y in g.coords]
            club_traces.append({
                "trace_id": int(row["trace_id"]),
                "name": row["name"],
                "coords": coords,
            })
        except Exception:
            continue
    print(f"{len(club_traces)} traces club chargées")
    return club_traces


# Pre-load
CLUB_TRACES = load_club_traces()

Chargement des traces club...
148 traces club chargées


## 4. Comparaison contre l'ensemble du club

In [6]:
def similarity_to_club(coords_test, club_traces, threshold_m=50, top_k=5):
    """Compare une trace test à toutes les traces club.
    
    Returns:
        dict avec:
        - global_score: similarité moyenne sur les top_k matches
        - best_matches: liste des top_k traces club les plus similaires
        - coverage: % de la trace test qui matche au moins UNE trace club
    """
    if not coords_test or not club_traces:
        return None
    
    pts_test = latlon_to_meters(coords_test)
    
    # Pour le coverage : union de tous les points club
    all_club_pts = []
    for t in club_traces:
        all_club_pts.extend(t["coords"])
    pts_club_all = latlon_to_meters(all_club_pts)
    tree_club_all = KDTree(pts_club_all)
    dists_test_to_any_club, _ = tree_club_all.query(pts_test, k=1)
    coverage = float((dists_test_to_any_club.flatten() < threshold_m).mean())
    
    # F1 individuel avec chaque trace
    matches = []
    for t in club_traces:
        r = f1_geo(coords_test, t["coords"], threshold_m)
        matches.append({
            "trace_id": t["trace_id"],
            "name": t["name"],
            "f1": r["f1"],
            "precision": r["precision"],
            "recall": r["recall"],
        })
    
    matches.sort(key=lambda x: -x["f1"])
    
    return {
        "coverage": coverage,
        "best_matches": matches[:top_k],
        "global_score": float(np.mean([m["f1"] for m in matches[:top_k]])),
        "n_compared": len(matches),
    }


# Test
"""
sample_route = ...  # un résultat de route() avec ['coords']
sim = similarity_to_club(sample_route["coords"], CLUB_TRACES)
print(f"Coverage : {sim['coverage']*100:.0f}%")
print(f"Top 5 traces club similaires :")
for m in sim["best_matches"]:
    print(f"  {m['name']} : F1 = {m['f1']*100:.0f}%")
"""
print("similarity_to_club() prête")

similarity_to_club() prête


## 5. Lecture de fichiers GPX

In [7]:
def read_gpx(filepath):
    """Lit un fichier GPX et retourne la liste des coords [(lat, lon), ...]."""
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        ns = {"gpx": "http://www.topografix.com/GPX/1/1"}
        coords = []
        for trkpt in root.iterfind(".//gpx:trkpt", ns):
            lat = float(trkpt.attrib["lat"])
            lon = float(trkpt.attrib["lon"])
            coords.append((lat, lon))
        # Fallback sans namespace
        if not coords:
            for trkpt in root.iter():
                if trkpt.tag.endswith("trkpt"):
                    lat = float(trkpt.attrib.get("lat", 0))
                    lon = float(trkpt.attrib.get("lon", 0))
                    if lat and lon:
                        coords.append((lat, lon))
        return coords
    except Exception as e:
        print(f"Erreur lecture GPX : {e}")
        return []


# Test
"""
coords = read_gpx("data/some_trace.gpx")
print(f"{len(coords)} points lus")
"""
print("read_gpx() prête")

read_gpx() prête


## 6. Test complet

In [8]:
# Comparer 2 traces club entre elles pour valider la métrique
if len(CLUB_TRACES) >= 2:
    t1, t2 = CLUB_TRACES[0], CLUB_TRACES[1]
    print(f"Comparaison '{t1['name']}' vs '{t2['name']}'")
    r = f1_geo(t1["coords"], t2["coords"], threshold_m=50)
    print(f"  Précision : {r['precision']*100:.0f}%")
    print(f"  Rappel    : {r['recall']*100:.0f}%")
    print(f"  F1        : {r['f1']*100:.0f}%")
    
    # Test similarité d'une trace club à TOUTES les autres
    print(f"\nSimilarité de '{t1['name']}' à l'ensemble du club :")
    others = [t for t in CLUB_TRACES if t['trace_id'] != t1['trace_id']]
    sim = similarity_to_club(t1["coords"], others)
    print(f"  Coverage : {sim['coverage']*100:.0f}%")
    print(f"  Top 3 traces similaires :")
    for m in sim["best_matches"][:3]:
        print(f"    {m['name']} : F1 = {m['f1']*100:.0f}%")

Comparaison '200 by CSP - Décembre' vs '200 Vexin'
  Précision : 0%
  Rappel    : 0%
  F1        : 0%

Similarité de '200 by CSP - Décembre' à l'ensemble du club :
  Coverage : 57%
  Top 3 traces similaires :
    DR 38 (81km, 423d+) : F1 = 53%
    Dr. 19+ (+Blandy) (116km, 583d+) : F1 = 48%
    Dr. 19+ (109km, 523d+) : F1 = 46%


## Notes

- Le seuil 50m fonctionne bien pour comparer des traces vélo
- Augmenter à 100m si tu veux être plus tolérant (utile pour la couverture globale)
- Diminuer à 20m si tu veux la similarité exacte
- Le KDTree rend les comparaisons rapides même pour 1000+ traces
